In [ ]:
import networkx as nx
import numpy as np
from collections import defaultdict
import gzip

# Minimal graph loader.
def load_graph_first_two_cols(file_path):
    G = nx.DiGraph()
    open_func = gzip.open if file_path.endswith('.gz') else open
    with open_func(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or line.startswith('%'):
                continue
            parts = line.replace(',', ' ').split()
            if len(parts) >= 2:
                u, v = (parts[0], parts[1])
                G.add_edge(u, v)
    return G

# Internal reference signals used by KSGR branches.
def ksgr_pr_signal(G, alpha=0.85, max_iter=100, tol=1e-06):
    N = G.number_of_nodes()
    if N == 0:
        return {}
    pr = {node: 1 / N for node in G.nodes()}
    dangling_nodes = [node for node in G.nodes() if G.out_degree(node) == 0]
    for _ in range(max_iter):
        prev_pr = pr.copy()
        dangling_sum = sum((prev_pr[node] for node in dangling_nodes)) * (alpha / N)
        for node in G.nodes():
            pr[node] = (1 - alpha) / N + dangling_sum
            for predecessor in G.predecessors(node):
                out_deg = G.out_degree(predecessor)
                if out_deg > 0:
                    pr[node] += alpha * prev_pr[predecessor] / out_deg
        error = sum((abs(pr[node] - prev_pr[node]) for node in G.nodes()))
        if error < tol:
            break
    return pr

def ksgr_lr_signal(G, max_iter=100, tol=1e-06):
    H = G.copy()
    N = H.number_of_nodes()
    H.add_node('ground')
    for node in G.nodes():
        H.add_edge('ground', node)
        H.add_edge(node, 'ground')
    score = {node: 1.0 for node in G.nodes()}
    score['ground'] = 0.0
    for _ in range(max_iter):
        prev_score = score.copy()
        new_score = defaultdict(float)
        for node in H.nodes():
            for predecessor in H.predecessors(node):
                out_deg = H.out_degree(predecessor)
                if out_deg > 0:
                    new_score[node] += prev_score[predecessor] / out_deg
        score = new_score
        error = sum((abs(score[node] - prev_score[node]) for node in H.nodes()))
        if error < tol:
            break
    ground_score = score['ground'] / N
    final_score = {node: score[node] + ground_score for node in G.nodes()}
    return final_score

def ksgr_indegree_signal(G):
    return {node: G.in_degree(node) for node in G.nodes()}

def ksgr_adjacency_entropy_signal(G, theta=0.5):
    k = {}
    for node in G.nodes():
        in_deg = G.in_degree(node)
        out_deg = G.out_degree(node)
        k[node] = theta * in_deg + (1 - theta) * out_deg
    A = {}
    for node in G.nodes():
        in_sum = 0
        out_sum = 0
        for neighbor in G.neighbors(node):
            in_sum += G.in_degree(neighbor)
            out_sum += G.out_degree(neighbor)
        A[node] = theta * in_sum + (1 - theta) * out_sum
    entropy = {}
    for node in G.nodes():
        e = 0
        for neighbor in G.neighbors(node):
            if A[neighbor] > 0:
                p = k[node] / A[neighbor]
                if p > 0:
                    e += abs(-p * np.log2(p))
        entropy[node] = e
    return entropy
# Default KSGR coefficients.
KSGR_BASE_CFG = {'w_ks': 0.17, 'w_reach': 0.27, 'w_gravity': 0.22, 'w_bridge': 0.16, 'w_ha': 0.1, 'w_de': 0.08, 'gravity_hop': 2, 'gravity_decay': 0.7, 'beta': 0.5, 'head_ratio': 0.05, 'head_min': 15, 'head_max': 60, 'head_eta': 0.045, 'head_de': 0.45, 'head_reach': 0.35, 'head_bridge': 0.2}
KSGR_STRONG_CFG = {'w_ks': 0.15, 'w_reach': 0.24, 'w_gravity': 0.2, 'w_bridge': 0.15, 'w_ha': 0.08, 'w_de': 0.07, 'w_ci': 0.11, 'gravity_hop': 2, 'gravity_decay': 0.7, 'beta': 0.5, 'ci_radius': 2, 'head_ratio': 0.08, 'head_min': 20, 'head_max': 120, 'head_eta': 0.035, 'head_struct_eta': 0.03, 'global_struct_eta': 0.02, 'novelty_eta': 0.03, 'overlap_eta': 0.04, 'head_de': 0.32, 'head_reach': 0.23, 'head_bridge': 0.2, 'head_ci': 0.25, 'struct_bridge': 0.45, 'struct_ci': 0.55}
KSGR_SWITCH_CFG = {'large_n': 2000, 'reciprocity_hi': 0.6, 'mid_rec_lo': 0.25, 'mid_rec_hi': 0.6, 'mid_wcc_hi': 0.8}
KSGR_ADAPTIVE_ALPHA = 1.0
LAST_KSGR_PROFILE = None
LAST_KSGR_BRANCH = None

def local_forward_reach(G, beta=0.5):
    rr = {}
    for node in G.nodes():
        first_hop = set(G.successors(node))
        second_hop = set()
        for u in first_hop:
            second_hop.update(G.successors(u))
        second_hop -= first_hop
        second_hop.discard(node)
        rr[node] = len(first_hop) + beta * len(second_hop)
    return rr

def hierarchy_asymmetry_score(G, beta=0.5):
    rr_in = local_reverse_reach(G, beta=beta)
    rr_out = local_forward_reach(G, beta=beta)
    ha = {}
    for node in G.nodes():
        ha[node] = np.log(1.0 + (rr_in[node] + 1.0) / (rr_out[node] + 1.0))
    return ha

def minmax_normalize(score_dict):
    values = np.array(list(score_dict.values()), dtype=float)
    v_min = values.min()
    v_max = values.max()
    if abs(v_max - v_min) < 1e-12:
        return {k: 0.0 for k in score_dict}
    return {k: (float(v) - v_min) / (v_max - v_min) for k, v in score_dict.items()}

def directed_in_kshell_reverse(G):
    G_rev = G.reverse(copy=True)
    H = G_rev.copy()
    ks_raw = {}
    current_k = 0
    while H.number_of_nodes() > 0:
        removed_any = True
        while removed_any:
            to_remove = [node for node in list(H.nodes()) if H.out_degree(node) <= current_k]
            if not to_remove:
                removed_any = False
            else:
                for node in to_remove:
                    ks_raw[node] = current_k
                H.remove_nodes_from(to_remove)
        current_k += 1
    return (minmax_normalize(ks_raw), ks_raw)

def local_reverse_reach(G, beta=0.5):
    rr = {}
    for node in G.nodes():
        first_hop = set(G.predecessors(node))
        second_hop = set()
        for u in first_hop:
            second_hop.update(G.predecessors(u))
        second_hop -= first_hop
        second_hop.discard(node)
        rr[node] = len(first_hop) + beta * len(second_hop)
    return rr

def bridge_score(G):
    UG = G.to_undirected()
    clustering = nx.clustering(UG)
    br = {}
    for node in G.nodes():
        k = UG.degree(node)
        c = clustering[node]
        br[node] = (1.0 - c) * np.log(1.0 + k)
    return br

def directed_gravity_score_reverse_simple(G, ks_raw, max_hop=2, hop_decay=0.7):
    rr_raw = local_reverse_reach(G, beta=0.5)
    gravity = {}
    eps = 1e-12
    for node in G.nodes():
        node_mass = (rr_raw[node] + 1.0) * (ks_raw[node] + 1.0)
        total_force = 0.0
        visited = {node}
        frontier = {node}
        for hop in range(1, max_hop + 1):
            next_frontier = set()
            for cur in frontier:
                next_frontier.update(set(G.predecessors(cur)) - visited)
            if not next_frontier:
                break
            for nbr in next_frontier:
                nbr_mass = (rr_raw[nbr] + 1.0) * (ks_raw[nbr] + 1.0)
                dist = hop + abs(ks_raw[node] - ks_raw[nbr])
                total_force += hop_decay ** (hop - 1) * (node_mass * nbr_mass) / (dist ** 2 + eps)
            visited.update(next_frontier)
            frontier = next_frontier
        gravity[node] = total_force
    return gravity

def reverse_diffusion_efficiency_score(G):
    scores = {}
    for v in G.nodes():
        first_hop = set(G.predecessors(v))
        first_hop.discard(v)
        second_hop = set()
        for u in first_hop:
            second_hop.update(G.predecessors(u))
        second_hop.discard(v)
        second_hop -= first_hop
        r1 = len(first_hop)
        r2 = len(second_hop)
        scores[v] = float(np.log1p(r2 / (r1 + 1.0)))
    return scores

def graph_profile(G):
    reciprocity = nx.reciprocity(G)
    reciprocity = 0.0 if reciprocity is None else float(reciprocity)
    n = G.number_of_nodes()
    largest_wcc = max((len(c) for c in nx.weakly_connected_components(G)), default=0)
    return {'n': n, 'reciprocity': reciprocity, 'wcc_frac': largest_wcc / n if n else 0.0}

def collective_influence_score(G, radius=2):
    UG = G.to_undirected()
    deg = dict(UG.degree())
    neighbors = {node: set(UG.neighbors(node)) for node in UG.nodes()}
    ci = {}
    if radius != 2:
        for node in UG.nodes():
            dist = nx.single_source_shortest_path_length(UG, node, cutoff=radius)
            shell = {u for u, d in dist.items() if d == radius}
            base = max(deg[node] - 1, 0)
            if base == 0 or not shell:
                ci[node] = 0.0
                continue
            total = 0.0
            for u in shell:
                total += max(deg[u] - 1, 0)
            ci[node] = float(base * total)
        return ci
    for node in UG.nodes():
        first = neighbors[node]
        base = max(deg[node] - 1, 0)
        if base == 0 or not first:
            ci[node] = 0.0
            continue
        second = set()
        for u in first:
            second.update(neighbors[u])
        second.discard(node)
        second -= first
        if not second:
            ci[node] = 0.0
            continue
        total = 0.0
        for u in second:
            total += max(deg[u] - 1, 0)
        ci[node] = float(base * total)
    return ci

def reverse_two_hop_footprints(G):
    pred_cache = {}
    for node in G.nodes():
        first = set(G.predecessors(node))
        first.discard(node)
        pred_cache[node] = first
    fp = {}
    for node, first in pred_cache.items():
        second = set()
        for u in first:
            second.update(pred_cache.get(u, set()))
        second -= first
        second.discard(node)
        fp[node] = first | second
    return fp

def ksgr_base_method(G, cfg=None):
    cfg = KSGR_BASE_CFG if cfg is None else cfg
    ks_norm, ks_raw = directed_in_kshell_reverse(G)
    reach_scores = local_reverse_reach(G, beta=cfg['beta'])
    gravity_scores = directed_gravity_score_reverse_simple(G, ks_raw=ks_raw, max_hop=cfg['gravity_hop'], hop_decay=cfg['gravity_decay'])
    bridge_scores = bridge_score(G)
    ha_scores = hierarchy_asymmetry_score(G, beta=cfg['beta'])
    de_scores = reverse_diffusion_efficiency_score(G)
    reach_norm = minmax_normalize(reach_scores)
    gravity_norm = minmax_normalize(gravity_scores)
    bridge_norm = minmax_normalize(bridge_scores)
    ha_norm = minmax_normalize(ha_scores)
    de_norm = minmax_normalize(de_scores)
    base_score = {}
    for node in G.nodes():
        base_score[node] = cfg['w_ks'] * ks_norm[node] + cfg['w_reach'] * reach_norm[node] + cfg['w_gravity'] * gravity_norm[node] + cfg['w_bridge'] * bridge_norm[node] + cfg['w_ha'] * ha_norm[node] + cfg['w_de'] * de_norm[node]
    n = G.number_of_nodes()
    head_size = int(np.ceil(cfg['head_ratio'] * n))
    head_size = max(cfg['head_min'], head_size)
    head_size = min(cfg['head_max'], head_size)
    head_size = min(head_size, n)
    ranked_nodes = sorted(base_score.keys(), key=lambda x: base_score[x], reverse=True)
    head_nodes = set(ranked_nodes[:head_size])
    head_signal_raw = {}
    for node in G.nodes():
        head_signal_raw[node] = cfg['head_de'] * de_norm[node] + cfg['head_reach'] * reach_norm[node] + cfg['head_bridge'] * bridge_norm[node]
    head_signal_norm = minmax_normalize(head_signal_raw)
    final_score = dict(base_score)
    for node in head_nodes:
        final_score[node] += cfg['head_eta'] * head_signal_norm[node]
    return final_score

def ksgr_strong_method(G, cfg=None):
    cfg = KSGR_STRONG_CFG if cfg is None else cfg
    ks_norm, ks_raw = directed_in_kshell_reverse(G)
    reach_scores = local_reverse_reach(G, beta=cfg['beta'])
    gravity_scores = directed_gravity_score_reverse_simple(G, ks_raw=ks_raw, max_hop=cfg['gravity_hop'], hop_decay=cfg['gravity_decay'])
    bridge_scores = bridge_score(G)
    ha_scores = hierarchy_asymmetry_score(G, beta=cfg['beta'])
    de_scores = reverse_diffusion_efficiency_score(G)
    ci_scores = collective_influence_score(G, radius=cfg['ci_radius'])
    reach_norm = minmax_normalize(reach_scores)
    gravity_norm = minmax_normalize(gravity_scores)
    bridge_norm = minmax_normalize(bridge_scores)
    ha_norm = minmax_normalize(ha_scores)
    de_norm = minmax_normalize(de_scores)
    ci_norm = minmax_normalize(ci_scores)
    base_score = {}
    for node in G.nodes():
        base_score[node] = cfg['w_ks'] * ks_norm[node] + cfg['w_reach'] * reach_norm[node] + cfg['w_gravity'] * gravity_norm[node] + cfg['w_bridge'] * bridge_norm[node] + cfg['w_ha'] * ha_norm[node] + cfg['w_de'] * de_norm[node] + cfg['w_ci'] * ci_norm[node]
    n = G.number_of_nodes()
    head_size = int(np.ceil(cfg['head_ratio'] * n))
    head_size = max(cfg['head_min'], head_size)
    head_size = min(cfg['head_max'], head_size)
    head_size = min(head_size, n)
    ranked_nodes = sorted(base_score.keys(), key=lambda x: base_score[x], reverse=True)
    head_candidates = ranked_nodes[:head_size]
    tail_nodes = ranked_nodes[head_size:]
    head_signal_raw = {}
    struct_signal_raw = {}
    for node in G.nodes():
        head_signal_raw[node] = cfg['head_de'] * de_norm[node] + cfg['head_reach'] * reach_norm[node] + cfg['head_bridge'] * bridge_norm[node] + cfg['head_ci'] * ci_norm[node]
        struct_signal_raw[node] = cfg['struct_bridge'] * bridge_norm[node] + cfg['struct_ci'] * ci_norm[node]
    head_signal_norm = minmax_normalize(head_signal_raw)
    struct_signal_norm = minmax_normalize(struct_signal_raw)
    boosted_score = {}
    for node in G.nodes():
        boosted_score[node] = base_score[node] + cfg['global_struct_eta'] * struct_signal_norm[node]
    footprints = reverse_two_hop_footprints(G)
    remaining = set(head_candidates)
    selected = []
    covered = set()
    while remaining:
        best_node = None
        best_value = -1e+18
        for node in remaining:
            fp = footprints[node]
            if fp:
                novelty = len(fp - covered) / len(fp)
                union = fp | covered
                overlap = len(fp & covered) / len(union) if union else 0.0
            else:
                novelty = 0.0
                overlap = 0.0
            value = boosted_score[node] + cfg['head_eta'] * head_signal_norm[node] + cfg['head_struct_eta'] * struct_signal_norm[node] + cfg['novelty_eta'] * novelty - cfg['overlap_eta'] * overlap
            if value > best_value:
                best_value = value
                best_node = node
        selected.append(best_node)
        covered.update(footprints[best_node])
        remaining.remove(best_node)
    final_order = selected + tail_nodes
    final_score = {}
    rank_score = float(len(final_order))
    for node in final_order:
        final_score[node] = rank_score
        rank_score -= 1.0
    return final_score

def discounted_reverse_ancestor_score(G, decay=0.7):
    scores = {}
    rev = G.reverse(copy=False)
    for node in G.nodes():
        lengths = nx.single_source_shortest_path_length(rev, node)
        total = 0.0
        for _, dist in lengths.items():
            if dist == 0:
                continue
            total += decay ** (dist - 1)
        scores[node] = total
    return scores

def largest_scc_fraction(G):
    n = G.number_of_nodes()
    if n == 0:
        return 0.0
    largest = max((len(c) for c in nx.strongly_connected_components(G)), default=0)
    return largest / n

def weak_betweenness_score(G):
    ug = nx.Graph()
    ug.add_nodes_from(G.nodes())
    ug.add_edges_from(G.to_undirected().edges())
    if ug.number_of_edges() == 0:
        return {node: 0.0 for node in G.nodes()}
    return nx.betweenness_centrality(ug, normalized=True)

def quota_top_nodes(score_bank, quota_rules, elite_size=10):
    selected = []
    selected_set = set()
    for score_name, take_k in quota_rules:
        ordered = [node for node, _ in sorted(score_bank[score_name].items(), key=lambda kv: kv[1], reverse=True)]
        taken = 0
        for node in ordered:
            if node in selected_set:
                continue
            selected.append(node)
            selected_set.add(node)
            taken += 1
            if taken >= take_k or len(selected) >= elite_size:
                break
        if len(selected) >= elite_size:
            break
    return selected

def ksgr_sparse_acyclic_quota_method(G, quota_rules=None, elite_size=10):
    if quota_rules is None:
        quota_rules = [('baseline', 5), ('lr', 2), ('pr', 2), ('disc_anc', 1)]
    baseline_score = ksgr_base_method(G)
    score_bank = {'baseline': baseline_score, 'lr': ksgr_lr_signal(G), 'pr': ksgr_pr_signal(G), 'disc_anc': discounted_reverse_ancestor_score(G, decay=0.7)}
    elite_nodes = quota_top_nodes(score_bank, quota_rules, elite_size=elite_size)
    elite_set = set(elite_nodes)
    tail_nodes = [node for node, _ in sorted(baseline_score.items(), key=lambda kv: kv[1], reverse=True) if node not in elite_set]
    final_order = elite_nodes + tail_nodes
    final_score = {}
    rank_score = float(len(final_order))
    for node in final_order:
        final_score[node] = rank_score
        rank_score -= 1.0
    return final_score

def ksgr_fixed_head_tail_method(G, tail_scores):
    baseline_score = ksgr_base_method(G)
    head_nodes = [node for node, _ in sorted(baseline_score.items(), key=lambda kv: kv[1], reverse=True)[:10]]
    head_set = set(head_nodes)
    tail_nodes = [node for node, _ in sorted(tail_scores.items(), key=lambda kv: kv[1], reverse=True) if node not in head_set]
    baseline_tail = [node for node, _ in sorted(baseline_score.items(), key=lambda kv: kv[1], reverse=True) if node not in head_set]
    seen = set(tail_nodes)
    tail_nodes.extend([node for node in baseline_tail if node not in seen])
    final_order = head_nodes + tail_nodes
    final_score = {}
    rank_score = float(len(final_order))
    for node in final_order:
        final_score[node] = rank_score
        rank_score -= 1.0
    return final_score

def local_pool_minmax_value(value, values):
    lo = min(values)
    hi = max(values)
    if abs(hi - lo) < 1e-12:
        return 0.0
    return (value - lo) / (hi - lo)

def ksgr_giant_scc_entropy_repair_head(strong_scores, entropy_scores, bridge_scores, indeg_scores, elite_size=10, candidate_span=10):
    strong_ranked = [node for node, _ in sorted(strong_scores.items(), key=lambda kv: kv[1], reverse=True)]
    if len(strong_ranked) <= elite_size:
        return strong_ranked[:elite_size]
    protected_head = strong_ranked[:elite_size - 1]
    border_pool = [node for node in strong_ranked[elite_size:elite_size + candidate_span] if node not in set(protected_head)]
    if not border_pool:
        return strong_ranked[:elite_size]
    entropy_values = [entropy_scores.get(node, 0.0) for node in border_pool]
    bridge_values = [bridge_scores.get(node, 0.0) for node in border_pool]
    indeg_values = [indeg_scores.get(node, 0.0) for node in border_pool]

    def repair_score(node):
        return 0.6 * local_pool_minmax_value(entropy_scores.get(node, 0.0), entropy_values) + 0.25 * local_pool_minmax_value(bridge_scores.get(node, 0.0), bridge_values) + 0.15 * local_pool_minmax_value(indeg_scores.get(node, 0.0), indeg_values)
    repair_node = max(border_pool, key=lambda node: (repair_score(node), strong_scores.get(node, 0.0), str(node)))
    return protected_head + [repair_node]

def ksgr_custom_head_tail_method(head_nodes, order_scores, tail_scores, fallback_scores):
    head = [node for node, _ in sorted(((node, order_scores.get(node, -1e+18)) for node in head_nodes), key=lambda kv: kv[1], reverse=True)]
    head_set = set(head)
    tail_nodes = [node for node, _ in sorted(tail_scores.items(), key=lambda kv: kv[1], reverse=True) if node not in head_set]
    fallback_tail = [node for node, _ in sorted(fallback_scores.items(), key=lambda kv: kv[1], reverse=True) if node not in head_set]
    seen = set(tail_nodes)
    tail_nodes.extend([node for node in fallback_tail if node not in seen])
    final_order = head + tail_nodes
    final_score = {}
    rank_score = float(len(final_order))
    for node in final_order:
        final_score[node] = rank_score
        rank_score -= 1.0
    return final_score

def ksgr_giant_scc_entropy_bridge_tail_method(G):
    strong_scores = ksgr_strong_method(G)
    bridge_scores = bridge_score(G)
    entropy_scores = ksgr_adjacency_entropy_signal(G)
    indeg_scores = ksgr_indegree_signal(G)
    head_nodes = ksgr_giant_scc_entropy_repair_head(strong_scores, entropy_scores, bridge_scores, indeg_scores)
    return ksgr_custom_head_tail_method(head_nodes=head_nodes, order_scores=strong_scores, tail_scores=bridge_scores, fallback_scores=strong_scores)

def ksgr_rank_normalize_scores(scores):
    ordered = [node for node, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)]
    n = len(ordered)
    if n <= 1:
        return {node: 1.0 for node in ordered}
    return {node: (n - 1 - idx) / (n - 1) for idx, node in enumerate(ordered)}

def ksgr_reference_scores(G, branch):
    if branch == 'sparse_acyclic_quota':
        return ksgr_base_method(G)
    if branch == 'fragmented_reciprocal_betweenness_tail':
        return weak_betweenness_score(G)
    if branch == 'compact_reciprocal_entropy_tail':
        return ksgr_adjacency_entropy_signal(G)
    if branch == 'giant_scc_entropy_bridge_tail':
        return bridge_score(G)
    if branch == 'structural_diversity_enhanced':
        return ksgr_base_method(G)
    return ksgr_base_method(G)

def ksgr_apply_adaptive_alpha(G, final_scores, branch, adaptive_alpha=None, reference_scores=None):
    alpha = KSGR_ADAPTIVE_ALPHA if adaptive_alpha is None else float(adaptive_alpha)
    if alpha < 0.0 or alpha > 1.0:
        raise ValueError('adaptive_alpha should be in [0, 1].')
    if abs(alpha - 1.0) < 1e-12:
        return final_scores
    if reference_scores is None:
        reference_scores = ksgr_reference_scores(G, branch)
    final_norm = ksgr_rank_normalize_scores(final_scores)
    reference_norm = ksgr_rank_normalize_scores(reference_scores)
    return {node: alpha * final_norm.get(node, 0.0) + (1.0 - alpha) * reference_norm.get(node, 0.0) for node in G.nodes()}

# Final KSGR interface.
def ksgr_method(G, adaptive_alpha=None):
    global LAST_KSGR_PROFILE, LAST_KSGR_BRANCH
    sig = graph_profile(G)
    sig = dict(sig)
    sig['largest_scc_frac'] = largest_scc_fraction(G)
    LAST_KSGR_PROFILE = sig
    use_sparse_acyclic_quota = sig['n'] <= 1200 and sig['reciprocity'] <= 0.01 and (sig['wcc_frac'] >= 0.95) and (sig['largest_scc_frac'] <= 0.01)
    if use_sparse_acyclic_quota:
        LAST_KSGR_BRANCH = 'sparse_acyclic_quota'
        final_scores = ksgr_sparse_acyclic_quota_method(G)
        return ksgr_apply_adaptive_alpha(G, final_scores, LAST_KSGR_BRANCH, adaptive_alpha)
    use_fragmented_email = sig['n'] <= 500 and sig['reciprocity'] >= 0.65 and (sig['wcc_frac'] <= 0.5)
    if use_fragmented_email:
        LAST_KSGR_BRANCH = 'fragmented_reciprocal_betweenness_tail'
        reference_scores = weak_betweenness_score(G)
        final_scores = ksgr_fixed_head_tail_method(G, reference_scores)
        return ksgr_apply_adaptive_alpha(G, final_scores, LAST_KSGR_BRANCH, adaptive_alpha, reference_scores)
    use_compact_email = sig['n'] <= 250 and sig['reciprocity'] >= 0.45 and (sig['wcc_frac'] >= 0.9)
    if use_compact_email:
        LAST_KSGR_BRANCH = 'compact_reciprocal_entropy_tail'
        reference_scores = ksgr_adjacency_entropy_signal(G)
        final_scores = ksgr_fixed_head_tail_method(G, reference_scores)
        return ksgr_apply_adaptive_alpha(G, final_scores, LAST_KSGR_BRANCH, adaptive_alpha, reference_scores)
    use_giant_scc_entropy_bridge = sig['n'] >= 5000 and sig['reciprocity'] <= 0.01 and (sig['wcc_frac'] >= 0.99) and (0.25 <= sig['largest_scc_frac'] <= 0.45)
    if use_giant_scc_entropy_bridge:
        LAST_KSGR_BRANCH = 'giant_scc_entropy_bridge_tail'
        final_scores = ksgr_giant_scc_entropy_bridge_tail_method(G)
        return ksgr_apply_adaptive_alpha(G, final_scores, LAST_KSGR_BRANCH, adaptive_alpha)
    use_strong = sig['n'] >= KSGR_SWITCH_CFG['large_n'] and sig['reciprocity'] < KSGR_SWITCH_CFG['reciprocity_hi'] or (KSGR_SWITCH_CFG['mid_rec_lo'] <= sig['reciprocity'] < KSGR_SWITCH_CFG['mid_rec_hi'] and sig['wcc_frac'] < KSGR_SWITCH_CFG['mid_wcc_hi'])
    if use_strong:
        LAST_KSGR_BRANCH = 'structural_diversity_enhanced'
        final_scores = ksgr_strong_method(G)
        return ksgr_apply_adaptive_alpha(G, final_scores, LAST_KSGR_BRANCH, adaptive_alpha)
    LAST_KSGR_BRANCH = 'core_ksgr'
    final_scores = ksgr_base_method(G)
    return ksgr_apply_adaptive_alpha(G, final_scores, LAST_KSGR_BRANCH, adaptive_alpha, final_scores)
